<a href="https://colab.research.google.com/github/shadmanr1/Milestone_1/blob/main/Week_3_Notebook_Shadman_Raakin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import (LinearRegression,Ridge,Lasso,ElasticNet)
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (mean_squared_error,mean_absolute_error,r2_score)
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving enhanced_box_office_data(2000-2024)u.csv to enhanced_box_office_data(2000-2024)u (1).csv


In [ ]:
df = pd.read_csv("enhanced_box_office_data(2000-2024)u.csv")
df.head()

,Rank,Release Group,$Worldwide,$Domestic,Domestic %,$Foreign,Foreign %,Year,Genres,Rating,Vote_Count,Original_Language,Production_Countries
0,1,Mission: Impossible II,546388108.0,215409889.0,39.4,330978219.0,60.6,2000,"Adventure, Action, Thriller",6.126/10,6741.0,en,United States of America
1,2,Gladiator,460583960.0,187705427.0,40.8,272878533.0,59.2,2000,"Action, Drama, Adventure",8.217/10,19032.0,en,"United Kingdom, United States of America"
2,3,Cast Away,429632142.0,233632142.0,54.4,196000000.0,45.6,2000,"Adventure, Drama",7.663/10,11403.0,en,United States of America
3,4,What Women Want,374111707.0,182811707.0,48.9,191300000.0,51.1,2000,"Comedy, Romance",6.45/10,3944.0,en,"United Kingdom, United States of America"
4,5,Dinosaur,349822765.0,137748063.0,39.4,212074702.0,60.6,2000,"Animation, Family, Adventure",6.544/10,2530.0,en,United States of America


Looking at which feature selection aprroach produces the best prediction of worldwide box office revenue?

Cleaning data:

In [ ]:
df = df.dropna()

In [ ]:
df = df.rename(columns={'$Worldwide':'Worldwide_Revenue'})
y = df['Worldwide_Revenue']
df['Rating'] = (df['Rating'].str.replace('/10','').astype(float))

In [ ]:
X = df[['Year','Rating','Vote_Count','Domestic %','Foreign %']]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Forward Selection

In [ ]:
lr = LinearRegression()

# Drop rows with NaN values from X_train and y_train
nan_rows = X_train.isnull().any(axis=1)
X_train_cleaned = X_train[~nan_rows]
y_train_cleaned = y_train[~nan_rows]

forward = SequentialFeatureSelector(lr,n_features_to_select='auto',direction='forward',cv=5)
forward.fit(X_train_cleaned, y_train_cleaned)
selected_forward = X.columns[forward.get_support()]
print(selected_forward)

Index(['Year', 'Vote_Count'], dtype='object')


In [ ]:
lr.fit(X_train[selected_forward],y_train)

pred_forward = lr.predict(X_test[selected_forward])

# Backward Selection

In [ ]:
backward = SequentialFeatureSelector(lr, n_features_to_select='auto',direction='backward',cv=5)

backward.fit(X_train, y_train)

selected_backward = X.columns[backward.get_support()]

print(selected_backward)

Index(['Year', 'Rating', 'Vote_Count'], dtype='object')


#PCA


In [ ]:
from sklearn.decomposition import PCA

pca = PCA()

X_train_pca = pca.fit_transform( X_train_scaled)

X_test_pca = pca.transform(X_test_scaled)

In [ ]:
print(
    np.cumsum(
        pca.explained_variance_ratio_
    )
)

[0.43317842 0.70170322 0.87101535 0.99999996 1.        ]


In [ ]:
pcr = LinearRegression()

pcr.fit(
    X_train_pca[:, :2],
    y_train
)

pred_pcr = pcr.predict(
    X_test_pca[:, :2]
)

#PLSR

In [ ]:
from sklearn.cross_decomposition import PLSRegression

pls = PLSRegression(n_components=2)

pls.fit(X_train_scaled,y_train)

pred_pls = pls.predict(X_test_scaled)

In [ ]:
pred_pls = pred_pls.flatten()

In [ ]:
r2_score(y_test, pred_pls)

np.sqrt(
    mean_squared_error(
        y_test,
        pred_pls
    )
)

np.float64(140926337.38601017)